In [3]:
def wrong_func(bytestring:bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

wrong_func("hello ".encode("utf-8"))

'hello '

In [ ]:
from .common import FIXTURES_PATH, gpt2_bytes_to_unicode

In [3]:

import time
from collections import Counter, defaultdict
from functools import partial, cmp_to_key
from multiprocessing import Pool

In [4]:
import struct
struct.calcsize("P") * 8

64

In [4]:
from typing import IO, Any, BinaryIO
import regex as re
import os

In [14]:
for m in re.finditer("(X)", "aXbXc"):
    print(m.group(), m.start(), m.end())

X 1 2
X 3 4


In [ ]:
it.

In [7]:
re.split("(X)", "aXXb")

['a', 'X', '', 'X', 'b']

In [6]:
def split_by_special_tokens(input_s: str,  special_tokens: list[str]):
    re_pattern = "|".join([re.escape(special_token) for special_token in special_tokens])
    results = re.split(re_pattern, input_s)
    return results

In [7]:
def split_by_pat(input_strings: list[str]):
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    results_unflatten = [re.findall(PAT, ss) for ss in input_strings]
    results_flattened = [item for sublist in results_unflatten for item in sublist]
    return results_flattened

In [8]:
class Token:
    """A token with an associated byte value and token ID."""
    # pair_token indicates this token is generated by two other token combined together
    def __init__(self, token_id: int, value:bytes, child_pair=None) -> None:
        self.token_id = token_id
        self.bytes = value
        self.child_pair = child_pair
        
    def to_string(self):
        return self.bytes.decode("utf-8", errors="ignore")

In [9]:
def tokens_to_string(tokens: list[Token]) -> str:
    return "".join(token.to_string() for token in tokens)
    

In [10]:
def cmp_id_tuple(t_a, t_b, vocab):
    b_a = vocab[t_a[0]] + vocab[t_a[1]]
    b_b = vocab[t_b[0]] + vocab[t_b[1]]
    if b_a < b_b:
        return -1
    elif b_a == b_b:
        return 0
    else:
        return 1

def countPair(word: list[Token]):
    counter = Counter()
    for i in range(len(word)-1):
        counter[(word[i].token_id, word[i+1].token_id)] +=1
    
    return counter



def replace_new_token(word: list[Token], new_token) -> list[Token]:
    new_word = []
    curInd = 0
    while curInd < len(word):
        if curInd < len(word) -1 and (word[curInd].token_id, word[curInd +1].token_id) == new_token.child_pair:
            new_word.append(new_token)
            curInd += 2
        else:
            new_word.append(word[curInd])
            curInd +=1

    return new_word
            
    


class TokenGroupCounter:
    def __init__(self, token_groups: list[list[Token]], counts: list[int], vocab:dict[int, bytes] ) -> None:
        self.token_groups = token_groups
        self.vocab = vocab
        self.counts = counts
        self.len = len(counts)
        
        # pair_counter is token id  tuple and count
        self.pair_counter = Counter()
        # Location mapping is from  pair of token id to a set of token_groups
        self.location_mapping = defaultdict(set)
        self.merges = []
        
        for i in range(self.len):
            tokens_i = self.token_groups[i]
            cnt = self.counts[i]
            for j in range(len(tokens_i)-1):
                tup = (tokens_i[j].token_id, tokens_i[j+1].token_id)
                self.pair_counter[tup] += cnt
                self.location_mapping[tup].add(i)      
        
    def iterate(self):
        most_common_pairs = self.pair_counter.most_common()
        top_frequency = most_common_pairs[0][1]
        top_pairs = [
          pair
          for pair, freq in most_common_pairs
          if freq == top_frequency
        ]

        comparator = partial(cmp_id_tuple, vocab = self.vocab)
        sorted_arr = sorted(top_pairs, key=cmp_to_key(comparator))
        merge_pair = sorted_arr[-1]
        self.merges.append((self.vocab[merge_pair[0]], self.vocab[merge_pair[1]]))
        new_token = Token(len(self.vocab), self.vocab[merge_pair[0]] + self.vocab[merge_pair[1]], child_pair = merge_pair)
        self.vocab[new_token.token_id] = new_token.bytes
        locations = self.location_mapping[merge_pair].copy()
        
        for location in locations:
            cnt = self.counts[location]
            before = self.token_groups[location]
            
            before_pair_counter = countPair(before)
            
            
            # need to implment replace_new_token
            after = replace_new_token(before, new_token)
            self.token_groups[location] =  after
            after_pair_counter = countPair(after)

            for pair in after_pair_counter:
                self.pair_counter[pair] += cnt * (after_pair_counter[pair] - before_pair_counter[pair])
                self.location_mapping[pair].add(location)                    
                    
            for pair in before_pair_counter:
                if pair not in after_pair_counter:
                    self.pair_counter[pair] -= cnt * before_pair_counter[pair]
                    self.location_mapping[pair].remove(location)

In [11]:
# input should be counter(str->int)
# output should be dict([list[token]], int))
def convert_to_inital_tokens(vocab:dict[int, bytes], str_list: Counter[str]):
    res = []
    counts = []
    for s, cnt in str_list.items():
        s_byte = s.encode('utf-8')
        tokens = [Token(int(b), bytes([b])) for b in s_byte]
        res.append(tokens)
        counts.append(cnt)
        
    return TokenGroupCounter(res, counts, vocab)

In [12]:
def pre_processing(initial_str: str, special_tokens: list[str]) -> TokenGroupCounter:
    splited_1 = split_by_special_tokens(test_str, special_tokens=special_tokens)
    splited_2 = split_by_pat(splited_1)
    return Counter(splited_2)

In [14]:
def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))


def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
    """Given the path to an input corpus, run train a BPE tokenizer and
    output its vocabulary and merges.

    Args:
        input_path (str | os.PathLike): Path to BPE tokenizer training data.
        vocab_size (int): Total number of items in the tokenizer's vocabulary (including special tokens).
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.
            These strings will never be split into multiple tokens, and will always be
            kept as a single token. If these special tokens occur in the `input_path`,
            they are treated as any other string.

    Returns:
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
            vocab:
                The trained tokenizer vocabulary, a mapping from int (token ID in the vocabulary)
                to bytes (token bytes)
            merges:
                BPE merges. Each list item is a tuple of bytes (<token1>, <token2>),
                representing that <token1> was merged with <token2>.
                Merges are ordered by order of creation.
    """
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    
    with open(input_path, "rb") as f:
        num_processes = 8
        boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

        # The following is a serial implementation, but you can parallelize this
        # by sending each start/end pair to a set of processes.
        chunks = [f.read(end - start).decode("utf-8", errors="ignore") for start, end in zip(boundaries[:-1], boundaries[1:])]
        worker = partial(pre_processing, special_tokens = special_tokens)
        with Pool(processes=num_processes) as pool:
            chunk_counts = pool.map(worker, chunks)
        
        print("hi")
        total_counts = Counter()
        for c in chunk_counts:
            total_counts.update(c)
        
        initial_vocab = {i: bytes([i]) for i in range(256)}
        state = convert_to_inital_tokens(initial_vocab, total_counts)
        for i in range(vocab_size - 256):
            print(i)
            state.iterate()
        
        
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            chunk = f.read(end - start).decode("utf-8", errors="ignore")
            # Run pre-tokenization on your chunk and store the counts for each pre-token
            # 1. splits all chunks by special_tokens, put special tokens into vocab
            # 2. count each chunk with adjcent token, 
            # 3. Merge all counts, find the next merge. Question: Is the merge added to the vocab? Yes
            # 4. Repeat. 
            # initial vaocab is just 0-255 single byte value. \
            state = create_initial_state(chunk, special_tokens)
            for i in range(vocab_size - 256):
                state.iterate()

In [ ]:
def test_train_bpe_speed():
    """
    Ensure that BPE training is relatively efficient by measuring training
    time on this small dataset and throwing an error if it takes more than 1.5 seconds.
    This is a pretty generous upper-bound, it takes 0.38 seconds with the
    reference implementation on my laptop. In contrast, the toy implementation
    takes around 3 seconds.
    """
    input_path = "tests/fixtures/corpus.en"
    start_time = time.time()
    _, _ = run_train_bpe(
        input_path=input_path,
        vocab_size=500,
        special_tokens=["<|endoftext|>"],
    )
    end_time = time.time()
    print(end_time - start_time)
    assert end_time - start_time < 1.5
test_train_bpe_speed()

In [233]:
def test_train_bpe():
    input_path = FIXTURES_PATH / "corpus.en"
    vocab, merges = run_train_bpe(
        input_path=input_path,
        vocab_size=500,
        special_tokens=["<|endoftext|>"],
    )

    # Path to the reference tokenizer vocab and merges
    reference_vocab_path = FIXTURES_PATH / "train-bpe-reference-vocab.json"
    reference_merges_path = FIXTURES_PATH / "train-bpe-reference-merges.txt"

    # Compare the learned merges to the expected output merges
    gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
    with open(reference_merges_path, encoding="utf-8") as f:
        gpt2_reference_merges = [tuple(line.rstrip().split(" ")) for line in f]
        reference_merges = [
            (
                bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
                bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
            )
            for merge_token_1, merge_token_2 in gpt2_reference_merges
        ]
    assert merges == reference_merges

    # Compare the vocab to the expected output vocab
    with open(reference_vocab_path, encoding="utf-8") as f:
        gpt2_reference_vocab = json.load(f)
        reference_vocab = {
            gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
            for gpt2_vocab_item, gpt2_vocab_index in gpt2_reference_vocab.items()
        }
    # Rather than checking that the vocabs exactly match (since they could
    # have been constructed differently), we'll make sure that the vocab keys and values match
    assert set(vocab.keys()) == set(reference_vocab.keys())
    assert set(vocab.values()) == set(reference_vocab.values())

In [198]:
test_str = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"
#test_str = "low low low low low"
splited_1 = split_by_special_tokens(test_str, special_tokens=["<|endoftext|>"])
#splited_2  = split_by_pat(splited_1)
splited_2 = test_str.split(' ')


In [199]:
print(splited_2)

['low', 'low', 'low', 'low', 'low', 'lower', 'lower', 'widest', 'widest', 'widest', 'newest', 'newest', 'newest', 'newest', 'newest', 'newest']


In [201]:
counted_splited = Counter(splited_2)

initial_vocab = {i: bytes([i]) for i in range(256)}
initial_tokens_counter = convert_to_inital_tokens(initial_vocab, counted_splited)

In [184]:
def token_groups_to_strings(token_groups: list[list[Token]]) -> list[str]:
  strings = ["".join(token.to_string() for token in group) for group in token_groups]
  print(strings)
  return strings
    
token_groups_to_strings(initial_tokens_counter.token_groups)

['low', 'lower', 'widest', 'newest']


['low', 'lower', 'widest', 'newest']

In [185]:
def default_map_to_char(counter):
    return [(bytes([item[0][0], item[0][1]]), item[1]) for item in counter.items()]
print(default_map_to_char(initial_tokens_counter.pair_counter))

[(b'lo', 7), (b'ow', 7), (b'we', 8), (b'er', 2), (b'wi', 3), (b'id', 3), (b'de', 3), (b'es', 9), (b'st', 9), (b'ne', 6), (b'ew', 6)]


In [186]:
print(default_map_to_char(initial_tokens_counter.location_mapping))

[(b'lo', {0, 1}), (b'ow', {0, 1}), (b'we', {1, 3}), (b'er', {1}), (b'wi', {2}), (b'id', {2}), (b'de', {2}), (b'es', {2, 3}), (b'st', {2, 3}), (b'ne', {3}), (b'ew', {3})]


In [187]:
initial_tokens_counter.iterate()

In [190]:
def print_location_mapping(location, vocab: dict[int, bytes]) -> None:
    for pair, groups in location.items():
        a = vocab[pair[0]].decode("utf-8", errors="ignore")
        b = vocab[pair[1]].decode("utf-8", errors="ignore")
        print(f"pair {pair} ({a!r}+{b!r}): {len(groups)} group(s)")
        for group in groups:
            print(f"    {group}")

In [191]:
print(print_location_mapping(initial_tokens_counter.location_mapping, initial_tokens_counter.vocab))

pair (108, 111) ('l'+'o'): 2 group(s)
    0
    1
pair (111, 119) ('o'+'w'): 2 group(s)
    0
    1
pair (119, 101) ('w'+'e'): 2 group(s)
    1
    3
pair (101, 114) ('e'+'r'): 1 group(s)
    1
pair (119, 105) ('w'+'i'): 1 group(s)
    2
pair (105, 100) ('i'+'d'): 1 group(s)
    2
pair (100, 101) ('d'+'e'): 1 group(s)
    2
pair (101, 115) ('e'+'s'): 0 group(s)
pair (115, 116) ('s'+'t'): 0 group(s)
pair (110, 101) ('n'+'e'): 1 group(s)
    3
pair (101, 119) ('e'+'w'): 1 group(s)
    3
pair (101, 256) ('e'+'st'): 2 group(s)
    2
    3
None


In [194]:
def print_pair_counter(pair_counter: dict[tuple[int, int], int], vocab: dict[int, bytes]) -> None:
    for pair, count in sorted(pair_counter.items(), key=lambda kv: kv[1], reverse=True):
        a = vocab[pair[0]].decode("utf-8", errors="ignore")
        b = vocab[pair[1]].decode("utf-8", errors="ignore")
        print(f"{pair} ({a!r}+{b!r}): {count}")
        
print_pair_counter(initial_tokens_counter.pair_counter, initial_tokens_counter.vocab)

(101, 256) ('e'+'st'): 9
(119, 101) ('w'+'e'): 8
(108, 111) ('l'+'o'): 7
(111, 119) ('o'+'w'): 7
(110, 101) ('n'+'e'): 6
(101, 119) ('e'+'w'): 6
(119, 105) ('w'+'i'): 3
(105, 100) ('i'+'d'): 3
(100, 101) ('d'+'e'): 3
(101, 114) ('e'+'r'): 2
(101, 115) ('e'+'s'): 0
(115, 116) ('s'+'t'): 0


In [202]:
def print_vocab(vocab: dict[int, bytes]) -> None:
    merged = [value.decode("utf-8", errors="ignore")
              for token_id, value in sorted(vocab.items())
              if token_id > 255]
    print(merged)
        

for i in range(6):
    initial_tokens_counter.iterate()
    print_vocab(initial_tokens_counter.vocab)

['st']
['st', 'est']
['st', 'est', 'ow']
['st', 'est', 'ow', 'low']
['st', 'est', 'ow', 'low', 'west']
['st', 'est', 'ow', 'low', 'west', 'ne']


In [138]:
" Sidney Rigdon ".split(" ")

['', 'Sidney', 'Rigdon', '']

In [15]:
input_path = FIXTURES_PATH / "corpus.en"
start_time = time.time()
_, _ = run_train_bpe(
    input_path=input_path,
    vocab_size=500,
    special_tokens=["<|endoftext|>"],
)
end_time = time.time()

iron cement is a ready for use paste which is laid as a fillet by putty knife or finger in the mould edges ( corners ) of the steel ingot mould .
iron cement protects the ingot against the hot , abrasive steel casting process .
a fire restant repair cement for fire places , ovens , open fireplaces etc .
construction and repair of highways and ...
an announcement must be commercial character .
goods and services advancement through the P.O.Box system is NOT ALLOWED .
deliveries ( spam ) and other improper information deleted .
translator Internet is a Toolbar for MS Internet Explorer .
it allows you to translate in real time any web pasge from one language to another .
you only have to select languages and TI does all the work for you ! automatic dictionary updates ....
this software is written in order to increase your English keyboard typing speed , through teaching the basics of how to put your hand on to the keyboard and give some training examples .
each lesson teaches some extra k

TypeError: cannot unpack non-iterable NoneType object

In [16]:
re.escape("|")

'\\|'